# AgingClockBench — Quickstart

Run a full benchmark comparison of PhenoAge and KDM on the bundled NHANES 1999-2000 sample in under 5 minutes.

**Install:** `pip install agingclockbench`

In [ ]:
from agingclockbench import PhenoAge, KDM, BenchmarkSuite
from agingclockbench.datasets import load_nhanes_sample
import matplotlib
%matplotlib inline

## 1. Load bundled NHANES data

NHANES 1999-2000 (N=4,086 complete cases) with 20-year mortality follow-up from NCHS.

In [ ]:
df = load_nhanes_sample()
print(f"{len(df)} participants | age {df.age.min():.0f}–{df.age.max():.0f} yr | "
      f"{df.mortstat.sum()} deaths ({df.mortstat.mean()*100:.1f}%)")
df.head()

## 2. Compute biological ages

In [ ]:
pa_result  = PhenoAge().transform(df)
kdm_result = KDM().transform(df)   # KDM doesn't need crp_mg_l

results = {'PhenoAge': pa_result, 'KDM': kdm_result}

for name, res in results.items():
    print(f"{name:12s} mean BA={res.biological_ages.mean():.1f} yr  "
          f"mean accel={res.accel.mean():.1f} yr")

## 3. Run benchmark suite (with mortality validation)

In [ ]:
suite = BenchmarkSuite(mortality_col='mortstat', followup_col='permth_exm')
report = suite.run(df, results)
report.to_dataframe()

**Reading the table:**
- `Pearson r` — how well biological age tracks chronological age (Levine 2018 target: ≥ 0.85 for PhenoAge)
- `Mort HR (per SD accel)` — hazard ratio per SD of age acceleration (higher = better predictor of mortality)
- `Mort p-value` — significance of the Cox PH coefficient

## 4. Visualize

In [ ]:
# Biological age vs chronological age scatter
fig1 = report.plot_comparison()
fig1.savefig('comparison.png', dpi=120, bbox_inches='tight')

In [ ]:
# Kaplan-Meier: survival by age-acceleration quartile
fig2 = report.plot_km_survival()
fig2.savefig('km_survival.png', dpi=120, bbox_inches='tight')

In [ ]:
# Inter-clock correlation heatmap
fig3 = report.plot_correlation_heatmap()
fig3.savefig('clock_correlations.png', dpi=120, bbox_inches='tight')

In [ ]:
# Full interactive HTML report
report.to_html('benchmark_report.html')
print('Open benchmark_report.html in your browser.')

## 5. Single patient example

In [ ]:
import pandas as pd

patient = pd.DataFrame([{
    'age': 52,
    'albumin_g_dl': 4.3,
    'creatinine_mg_dl': 0.9,
    'glucose_mg_dl': 87.0,
    'crp_mg_l': 0.3,
    'lymphocyte_pct': 28.0,
    'mcv_fl': 90.0,
    'rdw_pct': 13.0,
    'alp_u_l': 65.0,
    'wbc_k_ul': 6.0,
}])

res = PhenoAge().transform(patient)
print(f"Chronological age: {patient.age.iloc[0]:.0f} years")
print(f"Biological age:    {res.biological_ages.iloc[0]:.1f} years")
print(f"Age acceleration:  {res.accel.iloc[0]:.1f} years")